<a href="https://colab.research.google.com/github/Shaifali07/langgraph_learning/blob/main/basic_chatbot_using_langgraph_without_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.0 MB/s eta 0:00:00


In [48]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict, Annotated, List, Literal
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
import operator
from langgraph.graph.message import add_messages

In [ ]:
class Chatbot(TypedDict):
 messages:Annotated[list[BaseMessage],add_messages]

In [ ]:
from google.colab import userdata
import os

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

In [ ]:
llm=ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5)

In [50]:
def chat_bot(Chatbot):
    prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI bot to answer question of the users."),
    ("human", "Hello, I need help."),
    ("ai", "Hello! I am happy to help. what specific issues are you facing?"),
    ("human", "{user_input}"),
    ])
    chain=prompt|llm|StrOutputParser()
    response=chain.invoke({"user_input":Chatbot["messages"][-1]})
    return {"messages":[response]}


In [54]:
graph= StateGraph(Chatbot)

graph.add_node(chat_bot)

graph.add_edge(START,"chat_bot")
graph.add_edge("chat_bot",END)

workflow=graph.compile()

initial_state={
    "messages":["what is the capital of India"]
    }
final_state=workflow.invoke(initial_state)
# final_state=workflow.invoke({"messages":["Tell me nearest hill station near it"]})
print(final_state)
print(final_state["messages"][-1].content)

{'messages': [HumanMessage(content='what is the capital of India', additional_kwargs={}, response_metadata={}, id='7635ed75-744c-4d28-a2e6-95048b948441'), HumanMessage(content='The capital of India is New Delhi. Is there anything else I can help you with?', additional_kwargs={}, response_metadata={}, id='ae224fa4-0a28-49f8-9620-5e91a166c496')]}
The capital of India is New Delhi. Is there anything else I can help you with?


In [55]:
while(True):
  user_input = input("Type here: ")
  if (user_input.lower()=="quit"):
    break
  else:
    final_state=workflow.invoke({"messages":[user_input]})
    print(final_state["messages"][-1].content)


Type here: what is the capital of india
The capital of India is **New Delhi**. Is there anything else I can help you with?
Type here: how far delhi is from bom ba
It seems like you're trying to get the distance between Delhi and Bombay (now known as Mumbai). 

Delhi and Mumbai are approximately 1,414 kilometers (879 miles) apart. The flight duration between the two cities is around 2.5 hours. However, if you're traveling by road or train, the journey can take significantly longer, typically ranging from 20 to 24 hours depending on the mode of transportation and the route taken. 

If you have any other questions or need more specific information, feel free to ask.
Type here: quit
